In [1]:
# 1. Cài đặt 'uv' trước (nếu hệ thống chưa có)
!pip install uv -q

# 2. Chạy logic Python để lấy phiên bản
try: 
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil = f"pillow=={PIL.__version__}"
except ImportError: 
    _numpy = "numpy"
    _pil = "pillow"

# 3. Chạy lệnh cài đặt trên 1 dòng duy nhất để tránh lỗi khoảng trắng (backslash)
!uv pip install -qqq "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 46.8 MB/s eta 0:00:0000:01:00:01


In [2]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import os

max_seq_length = 2048
dtype = None # Auto 
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [3]:
data_path = "/kaggle/input/datasets/vutuanhungjjjj/phishing/train.jsonl"
dataset = load_dataset("json", data_files=data_path, split="train")

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}
"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/59976 [00:00<?, ? examples/s]

In [4]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length, 
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 1, 
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",
        report_to = "none",
        average_tokens_across_devices = False,
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/59976 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 59,976 | Num Epochs = 1 | Total steps = 3,749
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 12,156,928 of 3,224,906,752 (0.38% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,3.978981
20,3.161910
30,3.046087
40,2.656890
50,2.554263
60,2.733447
70,2.590050
80,2.711054
90,2.705153
100,2.727044


In [5]:
# !mkdir -p /kaggle/working/SLM_adapter
!ls /kaggle/working

huggingface_tokenizers_cache  outputs  unsloth_compiled_cache


In [6]:

OUTPUT_DIR = "/kaggle/working/SLM_adapter"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/SLM_adapter/tokenizer_config.json.


('/kaggle/working/SLM_adapter/tokenizer_config.json',
 '/kaggle/working/SLM_adapter/chat_template.jinja',
 '/kaggle/working/SLM_adapter/tokenizer.json')

# **Eval**

In [8]:
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

FastLanguageModel.for_inference(model)

test_path = "/kaggle/input/datasets/vutuanhungjjjj/phishing/test.jsonl"
if os.path.exists(test_path):
    test_dataset = load_dataset("json", data_files=test_path, split="train")
    
    y_true, y_pred = [], []
    eval_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""
    
    for item in tqdm(test_dataset, desc="Evaluating"):
        prompt = eval_prompt.format(item["instruction"], item["input"])
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=10,max_length=None, use_cache=True, pad_token_id=tokenizer.eos_token_id)
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        response = decoded.split("### Response:\n")[-1].strip()
        
        if "Phishing Email" in response or "Phishing" in response:
            pred = "Phishing Email"
        elif "Safe Email" in response or "Safe" in response:
            pred = "Safe Email"
        else:
            pred = "Unknown"
            
        y_true.append(item["output"].strip())
        y_pred.append(pred)
        
    print("\n--- Kết quả Đánh giá ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))
else:
    print("Không tìm thấy file test.jsonl")

Evaluating:  10%|▉         | 1226/12852 [08:04<1:07:39,  2.86it/s]Unsloth: Input IDs of shape torch.Size([1, 2332]) with length 2332 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Evaluating:  10%|▉         | 1263/12852 [08:19<1:10:51,  2.73it/s]Unsloth: Input IDs of shape torch.Size([1, 5134]) with length 5134 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Evaluating:  11%|█▏        | 1476/12852 [09:42<1:09:28,  2.73it/s]Unsloth: Input IDs of shape torch.Size([1, 6568]) with length 6568 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Evaluating:  11%|█▏        | 1477/12852 [09:43<1:56:14,  1.63it/s]Unsloth: Input IDs of shape torch.Size([1, 3634]) with length 3634 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if yo


--- Kết quả Đánh giá ---
Accuracy: 0.9833
                precision    recall  f1-score   support

Phishing Email       1.00      0.99      0.99      6426
    Safe Email       1.00      0.97      0.99      6426
       Unknown       0.00      0.00      0.00         0

      accuracy                           0.98     12852
     macro avg       0.66      0.66      0.66     12852
  weighted avg       1.00      0.98      0.99     12852



# Infer

In [9]:
FastLanguageModel.for_inference(model) # Đảm bảo bật chế độ inference nhanh

my_instruction = "Analyze the following email content and determine if it is a 'Safe Email' or a 'Phishing Email'."

my_email_content = """Dear Customer,
Your PayPal account has been temporarily suspended due to unusual activity. 
Please click the link below to verify your identity and restore access to your account:
http://paypal-security-update.com/login
Failure to do so within 24 hours will result in permanent termination of your account.

Sincerely,
PayPal Security Team"""

prompt = alpaca_prompt.format(my_instruction, my_email_content, "") 

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

print("Đang kiểm tra email...")
outputs = model.generate(**inputs, max_new_tokens=15, use_cache=True, pad_token_id=tokenizer.eos_token_id)
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

final_response = decoded_output.split("### Response:\n")[-1].strip()
print("\n--- Kết quả dự đoán ---")
print(f"Email này là: {final_response}")


Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Đang kiểm tra email...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



--- Kết quả dự đoán ---
Email này là: Phishing Email


In [35]:
!ls /kaggle/working

huggingface_tokenizers_cache  outputs	   SLM_adapter.zip
output			      SLM_adapter  unsloth_compiled_cache


In [19]:
!zip -r SLM_adapter.zip /kaggle/working/SLM_adapter 

  adding: kaggle/working/SLM_adapter/ (stored 0%)
  adding: kaggle/working/SLM_adapter/tokenizer.json (deflated 85%)
  adding: kaggle/working/SLM_adapter/README.md (deflated 65%)
  adding: kaggle/working/SLM_adapter/adapter_config.json (deflated 58%)
  adding: kaggle/working/SLM_adapter/tokenizer_config.json (deflated 96%)
  adding: kaggle/working/SLM_adapter/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/SLM_adapter/chat_template.jinja (deflated 71%)


In [23]:
!mkdir -p /kaggle/working/output
!cp /kaggle/working/SLM_adapter.zip /kaggle/working/output/

In [36]:
from IPython.display import FileLink

FileLink('/kaggle/working/SLM_adapter.zip')

/kaggle/working/SLM_adapter.zip